In [55]:
import re
import os

# ==========================================
# ISA DEFINITIONS
# ==========================================
R_OPCODE = "000000"
R_FUNCS = {
    "ADD": "0000", "SUB": "0001", "COMP": "0010", "CMP": "0010",
    "SHCMP": "0011", "STRCMP": "0100", "OR": "0101", 
    "LSH": "0110", "RSH": "0110", "XNOR": "1000", "AND": "1001", "SLT": "1010"
}

I_OPCODES = {"ADDI": "001000", "LW": "100011", "SW": "101011", "BEQ": "000100"}

def get_r(arm_reg):
    if 'sp' in arm_reg.lower(): return 'R29'
    if 'zr' in arm_reg.lower(): return 'R0'
    num = re.findall(r'\d+', arm_reg)
    return f"R{int(num[0]) + 1}" if num else "R0"

def r2b(reg_str):
    num = re.findall(r'\d+', reg_str)
    return format(int(num[0]), '05b') if num else "00000"

def get_imm(imm_str, bits=16):
    try:
        val = int(re.sub(r'[^0-9\-]', '', imm_str))
        if val < 0: val = (1 << bits) + val
        return format(val, f'0{bits}b')
    except: return "0" * bits

# ==========================================
# PHASE 1: COMPILER (.s -> .asm)
# ==========================================
def translate_gcc_to_asm(asm_lines):
    custom_asm = ["ADDI R29, R0, 0"]
    cmp_rs, cmp_rt = "R0", "R0"
    
    for line in asm_lines:
        line = line.split('//')[0].strip()
        if not line or (line.startswith('.') and not line.endswith(':')): continue
        if any(op in line for op in ['adr', 'sub\tsp', 'add\tsp', '.word', '.arch', '.file', '.ident', '.type', '.size', '.global']): 
            continue
        
        if line.endswith(':'):
            custom_asm.append(f"\n{line}")
            continue
        if 'ret' in line:
            # Control Hazard Padding for branches
            custom_asm.extend(["BEQ R0, R0, END_HALT", "NOP", "NOP", "NOP"])
            continue

        parts = [p.replace(',', '') for p in line.split()]
        op = parts[0].lower()
        new_insts = []

        if op == 'subs':
            rd, rs = get_r(parts[1]), get_r(parts[2])
            imm = int(parts[3].replace('#', ''))
            new_insts.append(f"ADDI {rd}, {rs}, -{imm}")
            cmp_rs, cmp_rt = rd, "R0"

        elif op in ['ldp', 'stp']:
            r1, r2 = get_r(parts[1]), get_r(parts[2])
            base = get_r(parts[3].strip('[]'))
            off = int(parts[4].strip(']').replace('#', '')) // 4 if len(parts) > 4 else 0
            if op == 'ldp':
                new_insts.extend([f"LW {r1}, {off}({base})", f"LW {r2}, {off+1}({base})"])
            else:
                new_insts.extend([f"SW {r1}, {off}({base})", f"SW {r2}, {off+1}({base})"])

        elif op in ['add', 'sub']:
            rd, rs = get_r(parts[1]), get_r(parts[2])
            if parts[3].replace('#', '').replace('-', '').isdigit():
                imm = int(parts[3].replace('#', ''))
                if op == 'sub': imm = -imm
                if imm % 4 == 0 and imm != 0: imm //= 4
                if 'sp' in parts[2].lower() and (rd == 'R1' or rd == 'R7'): imm = 0
                new_insts.append(f"ADDI {rd}, {rs}, {imm}")
            else:
                new_insts.append(f"{op.upper()} {rd}, {rs}, {get_r(parts[3])}")

        elif op == 'cmp':
            cmp_rs = get_r(parts[1])
            if parts[2].replace('#', '').replace('-', '').isdigit():
                val = parts[2].replace('#', '')
                new_insts.append(f"ADDI R28, R0, {val}")
                cmp_rt = "R28"
            else:
                cmp_rt = get_r(parts[2])
            new_insts.append(f"CMP {cmp_rs}, {cmp_rt}")

        elif op.startswith('b'):
            target = parts[1]
            if op == 'b': new_insts.append(f"BEQ R0, R0, {target}")
            elif op == 'beq': new_insts.append(f"BEQ {cmp_rs}, {cmp_rt}, {target}")
            elif op in ['bge', 'b.ge']: new_insts.extend([f"SLT R27, {cmp_rs}, {cmp_rt}", f"BEQ R27, R0, {target}"])
            elif op in ['ble', 'b.le']: new_insts.extend([f"SLT R27, {cmp_rt}, {cmp_rs}", f"BEQ R27, R0, {target}"])
                
            # Control Hazard Padding
            if 'b' in op: new_insts.extend(["NOP", "NOP", "NOP"])

        elif op == 'mov':
            rd = get_r(parts[1])
            val = parts[2].replace('#', '')
            if val.replace('-', '').isdigit(): new_insts.append(f"ADDI {rd}, R0, {val}")
            else: new_insts.append(f"ADD {rd}, R0, {get_r(val)}")

        custom_asm.extend(new_insts)

    custom_asm.extend(["\nEND_HALT:", "BEQ R0, R0, END_HALT", "NOP", "NOP", "NOP"])
    return custom_asm

# ==========================================
# PHASE 1.5: AUTOMATIC HAZARD RESOLVER
# ==========================================
def resolve_data_hazards(asm_lines):
    """Detects Read-After-Write hazards and injects 1 to 3 NOPs to simulate an HDU."""
    resolved_asm = []
    last_written = {} # Tracks the absolute index where each register was last written
    
    for inst in asm_lines:
        inst_clean = inst.split('//')[0].strip()
        
        # Keep labels, but clear tracking because branches break linear flow
        if not inst_clean or inst_clean.startswith('.') or inst_clean.endswith(':'):
            resolved_asm.append(inst)
            last_written = {} 
            continue
            
        parts = [p.replace(',', '').replace('(', ' ').replace(')', ' ') for p in inst_clean.split()]
        op = parts[0].upper()
        
        if op == "NOP":
            resolved_asm.append(inst)
            continue
            
        srcs = []
        dest = None
        
        # Identify Destination (Write) and Sources (Read)
        if op in ['ADD', 'SUB', 'SLT', 'OR', 'AND', 'XNOR']:
            if len(parts) >= 4:
                dest, srcs = parts[1], [parts[2], parts[3]]
        elif op == 'ADDI':
            if len(parts) >= 4:
                dest, srcs = parts[1], [parts[2]]
        elif op == 'LW':
            if len(parts) >= 4:
                dest, srcs = parts[1], [parts[3]] # LW Rd, imm(Rs)
        elif op == 'SW':
            if len(parts) >= 4:
                srcs = [parts[1], parts[3]] # SW Rt, imm(Rs)
        elif op in ['CMP', 'COMP']:
            if len(parts) >= 3:
                srcs = [parts[1], parts[2]]
        elif op == 'BEQ':
            if len(parts) >= 4:
                srcs = [parts[1], parts[2]]

        # Ignore R0 (hardwired zero)
        srcs = [s for s in srcs if s.startswith('R') and s != 'R0']
        if dest == 'R0': dest = None

        # Check for Hazards
        nops_needed = 0
        current_idx = len(resolved_asm)
        for src in srcs:
            if src in last_written:
                distance = current_idx - last_written[src]
                # A safe distance is 4 instructions apart (meaning 3 instructions between them)
                if distance <= 3:
                    nops_needed = max(nops_needed, 4 - distance)
        
        # Inject the NOPs
        for _ in range(nops_needed):
            resolved_asm.append("NOP")
        
        # Append the actual instruction
        resolved_asm.append(inst)
        
        # Log the write if applicable
        if dest and dest.startswith('R'):
            last_written[dest] = len(resolved_asm) - 1

    return resolved_asm

# ==========================================
# PHASE 2: ASSEMBLER (.asm -> .mem)
# ==========================================
def assemble_to_hex(custom_asm_lines):
    labels, final_code, pc = {}, [], 0
    for line in custom_asm_lines:
        line = line.split('//')[0].strip()
        if not line or line.endswith(':'):
            if line.endswith(':'): labels[line[:-1]] = pc
            continue
        final_code.append(line); pc += 1

    machine_code = []
    for cur_pc, inst in enumerate(final_code):
        if inst.upper() == "NOP": 
            machine_code.append("00000000")
            continue
            
        p = [x.replace(',', '') for x in inst.replace('(', ' ').replace(')', ' ').split()]
        op, bin_str = p[0].upper(), ""
        
        if op in R_FUNCS:
            if op in ['CMP', 'COMP']: rs, rt, rd = r2b(p[1]), r2b(p[2]), "00000"
            else: rd, rs, rt = r2b(p[1]), r2b(p[2]), r2b(p[3]) if len(p)>3 else "00000"
            bin_str = f"{R_OPCODE}{rs}{rt}{rd}0000000{R_FUNCS[op]}"
            
        elif op == 'ADDI':
            rs_src, rt_dest = r2b(p[2]), r2b(p[1])
            bin_str = f"{I_OPCODES[op]}{rs_src}{rt_dest}{get_imm(p[3])}"
            
        elif op in ['LW', 'SW']:
            rt_data, rs_base = r2b(p[1]), r2b(p[3])
            bin_str = f"{I_OPCODES[op]}{rs_base}{rt_data}{get_imm(p[2])}"
            
        elif op == 'BEQ':
            rs, rt = (r2b(p[1]), r2b(p[2])) if len(p)==4 else ("00000", "00000")
            target = p[-1]
            off = (labels[target] - cur_pc - 1) & 0xFFFF if target in labels else 0
            bin_str = f"{I_OPCODES['BEQ']}{rs}{rt}{format(off, '016b')}"
        
        machine_code.append(format(int(bin_str, 2), '08X') if bin_str else "00000000")
    return machine_code

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    FILENAME = "debug.asm" # Change to "debug.asm" if you edit manually
    
    if not os.path.exists(FILENAME):
        print(f"Error: '{FILENAME}' not found.")
    else:
        with open(FILENAME, "r") as f:
            input_lines = f.readlines()

        if FILENAME.endswith(".s"):
            print("Mode: Compiler (.s -> .asm -> .mem)")
            asm_code = translate_gcc_to_asm(input_lines)
        else:
            print("Mode: Assembler-Only (.asm -> .mem)")
            asm_code = input_lines

        # --- APPLY AUTOMATIC HAZARD PROTECTION ---
        asm_code = resolve_data_hazards(asm_code)

        # Write the protected assembly to debug.asm so you can verify the NOPs
        with open("debug.asm", "w") as f:
            f.write("\n".join(asm_code))
        if FILENAME.endswith(".s"):
            print("-> Generated 'debug.asm' (Check this file to see where NOPs were injected!)")

        # Assemble the final protected code
        hex_list = assemble_to_hex(asm_code)
        
        with open("machine_code.mem", "w") as f:
            for h in hex_list:
                f.write(h + ",\n") 
                
        print(f"-> Generated 'machine_code.mem' with {len(hex_list)} instructions.")

Mode: Assembler-Only (.asm -> .mem)
-> Generated 'machine_code.mem' with 60 instructions.
